# Customer Churn Prediction & Retention Analysis
**Prepared by: Pranjal Jain**

Educational project using a synthetic subscription-customer dataset; no real customer data is included.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")
RANDOM_STATE=42

## 1. Load and inspect

In [ ]:
raw_df=pd.read_csv("customer_churn_raw.csv")
print("Shape:",raw_df.shape); display(raw_df.head()); raw_df.info()
display(raw_df.describe(include="all").T)
print("Missing values:\\n",raw_df.isna().sum())
print("Duplicates:",raw_df.duplicated().sum())
display(raw_df["Churn"].value_counts(normalize=True).rename("Proportion"))

## 2. Clean and explore

In [ ]:
df=raw_df.drop_duplicates().copy()
df["MonthlyCharges"]=pd.to_numeric(df["MonthlyCharges"],errors="coerce")
df["TenureMonths"]=pd.to_numeric(df["TenureMonths"],errors="coerce")
df["ChurnFlag"]=df["Churn"].map({"No":0,"Yes":1})
print("Before:",len(raw_df),"After duplicate removal:",len(df))
df.to_csv("customer_churn_cleaned.csv",index=False)
fig,axes=plt.subplots(2,3,figsize=(16,9))
sns.countplot(data=df,x="Churn",ax=axes[0,0]); axes[0,0].set_title("Churn distribution")
sns.histplot(data=df,x="TenureMonths",hue="Churn",multiple="stack",ax=axes[0,1]); axes[0,1].set_title("Tenure by churn")
sns.boxplot(data=df,x="Churn",y="MonthlyCharges",ax=axes[0,2]); axes[0,2].set_title("Monthly charges by churn")
sns.countplot(data=df,x="Contract",hue="Churn",ax=axes[1,0]); axes[1,0].set_title("Contract and churn")
sns.countplot(data=df,x="TechSupport",hue="Churn",ax=axes[1,1]); axes[1,1].set_title("Tech support and churn")
sns.countplot(data=df,x="PaymentMethod",hue="Churn",ax=axes[1,2]); axes[1,2].tick_params(axis="x",rotation=20); axes[1,2].set_title("Payment method and churn")
plt.tight_layout(); plt.show()

## 3. Preprocess and split

In [ ]:
X=df.drop(columns=["CustomerID","Churn","ChurnFlag"]); y=df["ChurnFlag"].astype(int)
num_cols=X.select_dtypes(include=np.number).columns.tolist()
cat_cols=X.select_dtypes(exclude=np.number).columns.tolist()
prep=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num_cols),
("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=RANDOM_STATE,stratify=y)
print("Train:",len(X_train),"Test:",len(X_test),"Churn rate:",round(y.mean(),3))

## 4. Train and compare three models

In [ ]:
estimators={
"Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE),
"Decision Tree":DecisionTreeClassifier(max_depth=5,min_samples_leaf=10,class_weight="balanced",random_state=RANDOM_STATE),
"Random Forest":RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=5,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1)}
models={}; rows=[]
for name,est in estimators.items():
    pipe=Pipeline([("prep",prep),("model",est)]); pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); score=pipe.predict_proba(X_test)[:,1]; models[name]=pipe
    rows.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Precision (churn)":precision_score(y_test,pred,zero_division=0),"Recall (churn)":recall_score(y_test,pred,zero_division=0),"F1 (churn)":f1_score(y_test,pred,zero_division=0),"ROC-AUC":roc_auc_score(y_test,score)})
results=pd.DataFrame(rows).sort_values("ROC-AUC",ascending=False); display(results.round(3))
best_name=results.iloc[0]["Model"]; best_model=models[best_name]
print("Selected by ROC-AUC:",best_name)
print(classification_report(y_test,best_model.predict(X_test),target_names=["No churn","Churn"],zero_division=0))

## 5. Confusion matrix and ROC curve

In [ ]:
ConfusionMatrixDisplay.from_estimator(best_model,X_test,y_test,display_labels=["No churn","Churn"],values_format="d")
plt.title("Confusion matrix — "+best_name); plt.tight_layout(); plt.show()
RocCurveDisplay.from_estimator(best_model,X_test,y_test); plt.title("ROC curve — "+best_name); plt.tight_layout(); plt.show()

## 6. Feature importance

In [ ]:
perm=permutation_importance(best_model,X_test,y_test,n_repeats=8,random_state=RANDOM_STATE,scoring="roc_auc")
importance=pd.DataFrame({"Feature":X_test.columns,"MeanImportance":perm.importances_mean,"Std":perm.importances_std}).sort_values("MeanImportance",ascending=False)
display(importance)
sns.barplot(data=importance.head(10),x="MeanImportance",y="Feature"); plt.title("Permutation importance (ROC-AUC)"); plt.tight_layout(); plt.show()

## 7. Customer churn risk prediction function

In [ ]:
def predict_churn_risk(tenure_months,monthly_charges,contract,internet_service,tech_support,payment_method,senior_citizen="No",paperless_billing="Yes"):
    customer=pd.DataFrame([{"TenureMonths":tenure_months,"MonthlyCharges":monthly_charges,"Contract":contract,"InternetService":internet_service,"TechSupport":tech_support,"PaymentMethod":payment_method,"SeniorCitizen":senior_citizen,"PaperlessBilling":paperless_billing}])
    p=float(best_model.predict_proba(customer)[0,1])
    return {"risk_level":"Higher predicted risk" if p>=.5 else "Lower predicted risk","estimated_probability":round(p,3)}
predict_churn_risk(4,105,"Month-to-month","Fiber optic","No","Electronic check")

## 8. Retention recommendations and limitations
- For short-tenure customers, improve onboarding and offer a helpful early check-in.
- For month-to-month contracts, explain longer-term options transparently without pressure.
- For high monthly charges, review plan fit and explain available lower-cost options.
- For customers without technical support, make support and troubleshooting easy to access.
- For payment friction, offer secure payment choices and clear billing reminders.

**Interpretation:** permutation importance shows model reliance, not causation. The synthetic data generator influences which patterns appear. Actual metrics are produced when the notebook runs. This is not the official IBM Telco dataset and must not be represented as real customer performance. Real deployment requires representative data, privacy controls, fairness checks, calibration, appropriate thresholds, and drift monitoring.